# EMA 6938 — Data Science for Materials
## Week 1 Lab Notebook: Introduction to Materials Data Science

**Name:** *(your name here)*  
**Date:** *(date)*  
**Kernel:** Python (matds)

---

### About this notebook

This notebook has five parts:

| Part | Title | When | Points |
|------|-------|------|--------|
| A | Environment verification | In-class, Day 1 | Completion |
| B | Python primer | In-class, Day 1 | Completion |
| C | NumPy & pandas with materials data | In-class, Day 1 | Completion |
| D | Materials Project API | In-class, Day 1 | Completion |
| E | Take-home reflection | Independent, due Sunday 11:59 PM | Graded |

**Parts A–D** are completed during the Day 1 lab session (75 min). Follow along with the instructor's screenshare.  
**Part E** is take-home — complete it independently before Sunday 11:59 PM.

**Submission:** Upload this `.ipynb` file to Canvas. Run `Kernel → Restart & Run All` before submitting to confirm all cells execute cleanly.

> **AI tool disclosure:** If you used any AI coding assistant (GitHub Copilot, ChatGPT, etc.) while completing this notebook, describe briefly which tool, for what purpose, and what you verified yourself. Delete this line if no AI tools were used.

---

## Part A — Environment Verification
*(In-class — do not modify these cells)*

Run each cell in order. If any cell raises an `ImportError`, post the error message in the **General Course Questions** discussion thread on Canvas immediately — do not wait until the deadline.

In [ ]:
# Cell A1 — Verify Python version
import sys
print(f"Python version: {sys.version}")
assert sys.version_info >= (3, 10), "Python 3.10+ required. Please reinstall the matds environment."
print("✅ Python version OK")

In [ ]:
# Cell A2 — Verify all core library imports
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pymatgen.core

packages = {
    "NumPy":     np.__version__,
    "pandas":    pd.__version__,
    "matplotlib": matplotlib.__version__,
    "pymatgen":  pymatgen.core.__version__,
}

print("Package versions:")
for pkg, ver in packages.items():
    print(f"  {pkg:<15} {ver}")

print("\n✅ All core imports successful")

In [ ]:
# Cell A3 — Verify Materials Project API key
from dotenv import load_dotenv
import os

load_dotenv()  # Reads .env file from repository root
API_KEY = os.getenv("MP_API_KEY")

if API_KEY:
    print(f"✅ MP API key loaded ({len(API_KEY)} characters)")
    print("   Key preview:", API_KEY[:4] + "*" * (len(API_KEY) - 8) + API_KEY[-4:])
else:
    print("❌ MP API key not found.")
    print("   Create a .env file in the repository root with:")
    print("   MP_API_KEY=your_actual_key_here")
    print("   Register for a free key at: https://next.materialsproject.org/api")

In [ ]:
# Cell A4 — Test Materials Project API connection
# Note: API key must be active (can take up to 1 hour after registration)
# If this cell fails, use the Binder fallback link posted in the Canvas Week 1 module

from mp_api.client import MPRester

try:
    with MPRester(API_KEY) as mpr:
        # Minimal test query — fetches one entry
        result = mpr.materials.summary.search(material_ids=["mp-5020"], fields=["formula_pretty"])
        print(f"✅ MP API connection successful")
        print(f"   Test query returned: {result[0].formula_pretty}")
except Exception as e:
    print(f"❌ MP API connection failed: {e}")
    print("   If your key is new, wait up to 1 hour and retry.")
    print("   Use the Binder fallback link in Canvas in the meantime.")

---
## Part B — Python Primer
*(In-class — complete each cell, then answer the Tasks in markdown)*

All examples use materials science context. Every data structure here reappears in later weeks.

In [ ]:
# Cell B1 — Dictionaries: element properties

elements = {
    'Fe': {'atomic_mass': 55.845, 'melting_point_K': 1811, 'crystal_structure': 'BCC'},
    'Al': {'atomic_mass': 26.982, 'melting_point_K': 933,  'crystal_structure': 'FCC'},
    'Ti': {'atomic_mass': 47.867, 'melting_point_K': 1941, 'crystal_structure': 'HCP'},
}

# Example: access Fe melting point
print(f"Fe melting point: {elements['Fe']['melting_point_K']} K")

# Example: iterate over all elements
print("\nAll elements:")
for symbol, props in elements.items():
    print(f"  {symbol}: {props['crystal_structure']}, mp = {props['melting_point_K']} K")

In [ ]:
# Cell B1 — TASK
# Add Cu (atomic_mass=63.546, melting_point_K=1358, crystal_structure='FCC')
# and Ni (atomic_mass=58.693, melting_point_K=1728, crystal_structure='FCC')
# to the elements dictionary, then print the updated dictionary.

# Your code here:
elements = {
    'Fe': {'atomic_mass': 55.845, 'melting_point_K': 1811, 'crystal_structure': 'BCC'},
    'Al': {'atomic_mass': 26.982, 'melting_point_K': 933,  'crystal_structure': 'FCC'},
    'Ti': {'atomic_mass': 47.867, 'melting_point_K': 1941, 'crystal_structure': 'HCP'},
    'Cu': {'atomic_mass': 63.546, 'melting_point_K': 1358, 'crystal_structure': 'FCC'},
    'Ni': {'atomic_mass': 58.693, 'melting_point_K': 1728, 'crystal_structure': 'FCC'},
}


print("\nAll elements:")
for symbol, props in elements.items():
    print(f"  {symbol}: {props['crystal_structure']}, mp = {props['melting_point_K']} K")

In [ ]:
# Cell B2 — Loops and list comprehensions: materials property lists

# Standard loop
melting_points = []
for symbol, props in elements.items():
    melting_points.append(props['melting_point_K'])
print("Melting points (loop):", melting_points)

# Equivalent list comprehension — more Pythonic
melting_points_lc = [props['melting_point_K'] for props in elements.values()]
print("Melting points (list comp):", melting_points_lc)

In [ ]:
# Cell B2 — TASK
# Using a list comprehension, create a list of (symbol, melting_point_K) tuples
# for elements with melting point above 1500 K.
# Expected output (order may vary): [('Fe', 1811), ('Ti', 1941), ('Ni', 1728)]

# Your code here:
refractory = [symbol for symbol in elements.items() props['melting_point_K'] for props in elements.values()]
print("Elements with mp > 1500 K:", refractory)

In [ ]:
# Cell B3 — Functions: unit conversions for materials data

def celsius_to_kelvin(T_C: float) -> float:
    """Convert temperature from Celsius to Kelvin."""
    return T_C + 273.15

def kelvin_to_celsius(T_K: float) -> float:
    """Convert temperature from Kelvin to Celsius."""
    return T_K - 273.15

# Test
print(f"Room temperature: {celsius_to_kelvin(25):.2f} K")
print(f"Fe melting point: {kelvin_to_celsius(1811):.2f} °C")

In [ ]:
# Cell B3 — TASK
# Write a function eV_to_kJ_per_mol(energy_eV) that converts formation energy
# from eV/atom to kJ/mol.
# Conversion: 1 eV = 96.485 kJ/mol
#
# Then use it to convert the following typical DFT formation energies:
# Fe2O3: -1.67 eV/atom
# Al2O3: -3.43 eV/atom
# TiO2:  -3.11 eV/atom

# Your code here:
def eV_to_kJ_per_mol(energy_eV):
    pass  # replace with your implementation

for material, energy in [('Fe2O3', -1.67), ('Al2O3', -3.43), ('TiO2', -3.11)]:
    print(f"{material}: {eV_to_kJ_per_mol(energy):.1f} kJ/mol")

---
## Part C — NumPy & Pandas with Materials Data
*(In-class — run each cell, complete tasks)*

We use a small sample dataset of Materials Project entries. The CSV contains bandgap and formation energy values for 50 binary oxides.

In [ ]:
# Cell C1 — Load and inspect the sample dataset
import pandas as pd
import numpy as np

df = pd.read_csv('week1_materials_sample.csv')

print(f"Shape: {df.shape}  ({df.shape[0]} materials, {df.shape[1]} columns)")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Cell C2 — Statistical summary
print("Statistical summary:")
df.describe().round(3)

In [ ]:
# Cell C2 — TASK
# 1. How many unique elements appear across all compositions in the dataset?
#    Hint: look at the 'elements' column — it stores element lists as strings.
# 2. How many materials have a bandgap of exactly 0 eV (i.e., are metallic)?
# 3. What is the mean formation energy of materials with bandgap > 3 eV?

# Your code here:

# 1. Unique elements

# 2. Metallic materials (bandgap == 0)

# 3. Mean formation energy of wide-bandgap materials


In [ ]:
# Cell C3 — NumPy operations on property arrays

bandgaps = df['band_gap'].values  # Extract as NumPy array
form_energies = df['formation_energy_per_atom'].values

print(f"Bandgap array type: {type(bandgaps)}")
print(f"Min bandgap:  {np.min(bandgaps):.3f} eV")
print(f"Max bandgap:  {np.max(bandgaps):.3f} eV")
print(f"Mean bandgap: {np.mean(bandgaps):.3f} eV")
print(f"Std bandgap:  {np.std(bandgaps):.3f} eV")
print(f"\nPearson correlation (bandgap vs. formation energy): {np.corrcoef(bandgaps, form_energies)[0,1]:.3f}")

In [ ]:
# Cell C4 — Scatter plot: bandgap vs. formation energy
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))

scatter = ax.scatter(
    df['band_gap'],
    df['formation_energy_per_atom'],
    alpha=0.7,
    edgecolors='white',
    linewidth=0.5,
    s=60
)

ax.set_xlabel('Band Gap (eV)', fontsize=12)
ax.set_ylabel('Formation Energy (eV/atom)', fontsize=12)
ax.set_title('Band Gap vs. Formation Energy — MP Binary Oxides Sample', fontsize=13)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('week1_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as week1_scatter.png")

In [ ]:
# Cell C4 — TASK
# The scatter plot above uses a single color for all points.
# Modify it to add a third dimension: color each point by its
# 'volume_per_atom' value using a colormap (e.g. 'viridis').
# Add a colorbar with the label 'Volume per atom (Å³)'.
#
# Hint: in scatter(), use c=df['volume_per_atom'] and cmap='viridis',
# then use plt.colorbar() on the scatter object.

# Your code here:


---
## Part D — Materials Project API
*(In-class — follow along with the instructor)*

You already tested the API connection in Part A. Now you will use it to retrieve real crystal structures and explore what information is available.

In [ ]:
# Cell D1 — Your first real API call: BaTiO3
from mp_api.client import MPRester
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv("MP_API_KEY")

with MPRester(API_KEY) as mpr:
    struct = mpr.get_structure_by_material_id("mp-5020")  # BaTiO3 — tetragonal

print("Structure retrieved!")
print(struct)

In [ ]:
# Cell D2 — Inspect the Structure object

print(f"Formula:       {struct.formula}")
print(f"Reduced:       {struct.reduced_formula}")
print(f"Space group:   {struct.get_space_group_info()}")
print(f"Num sites:     {len(struct)}")
print(f"Lattice (Å):   a={struct.lattice.a:.4f}, b={struct.lattice.b:.4f}, c={struct.lattice.c:.4f}")
print(f"Volume (Å³):   {struct.volume:.4f}")
print()
print("Atomic sites (fractional coordinates):")
for site in struct:
    print(f"  {site.species_string:<4}  {site.frac_coords}")

In [ ]:
# Cell D3 — Retrieve computed properties alongside the structure

with MPRester(API_KEY) as mpr:
    results = mpr.summary.search(
        material_ids=["mp-5020"],
        fields=["formula_pretty", "band_gap", "formation_energy_per_atom",
                "energy_above_hull", "is_stable", "volume", "density"]
    )

entry = results[0]
print(f"Material:                 {entry.formula_pretty}")
print(f"Band gap:                 {entry.band_gap:.3f} eV")
print(f"Formation energy:         {entry.formation_energy_per_atom:.4f} eV/atom")
print(f"Energy above hull:        {entry.energy_above_hull:.4f} eV/atom")
print(f"Thermodynamically stable: {entry.is_stable}")
print(f"Volume:                   {entry.volume:.3f} Å³")
print(f"Density:                  {entry.density:.3f} g/cm³")

In [ ]:
# Cell D4 — Explore: change the material ID

# Try fetching a different material.
# Some suggestions to try (remove the # and run):
# mp-1234    — SiO2 (quartz)
# mp-2133    — Fe2O3 (hematite)
# mp-22526   — NaCl (halite/rocksalt)
# mp-2534    — Al2O3 (corundum)

MATERIAL_ID = "mp-1234"  # Change this to explore

with MPRester(API_KEY) as mpr:
    struct2 = mpr.get_structure_by_material_id(MATERIAL_ID)
    results2 = mpr.summary.search(
        material_ids=[MATERIAL_ID],
        fields=["formula_pretty", "band_gap", "formation_energy_per_atom", "is_stable"]
    )

e = results2[0]
print(f"Material ID:     {MATERIAL_ID}")
print(f"Formula:         {e.formula_pretty}")
print(f"Band gap:        {e.band_gap:.3f} eV")
print(f"Formation energy:{e.formation_energy_per_atom:.4f} eV/atom")
print(f"Stable:          {e.is_stable}")
print(f"Space group:     {struct2.get_space_group_info()}")

In [ ]:
# Cell D5 — Nearest neighbors: who is bonded to whom?

# Get nearest neighbors of the first site in BaTiO3
site_index = 0
site = struct[site_index]
neighbors = struct.get_neighbors(site, r=3.5)  # within 3.5 Angstrom

print(f"Central atom: {site.species_string} at {site.frac_coords.round(3)}")
print(f"Neighbors within 3.5 Å:")
for nbr in sorted(neighbors, key=lambda x: x.nn_distance):
    print(f"  {nbr.species_string:<4}  distance = {nbr.nn_distance:.4f} Å")

---
## Part E — Take-Home Reflection
*(Complete independently — due Sunday 11:59 PM)*

These cells require written answers (in markdown) and extended code. Read all three questions before starting — they build on each other.

**Grading:** Part E is the only graded section. It is assessed on correctness of code, quality of scientific reasoning, and clarity of written responses.

### E1 — The Pymatgen Structure object

**Question:** In 2–3 sentences, explain what the Pymatgen `Structure` object represents physically. What information does it contain that a simple chemical formula (e.g. BaTiO₃) does not? Why does this extra information matter for machine learning in materials science?

**Your answer:** *(write here — double-click to edit)*



### E2 — Improve the scatter plot

**Task:** Return to the scatter plot from Cell C4 (bandgap vs. formation energy). Identify **two scientific improvements** and implement them. In a markdown cell below your improved plot, explain what you changed and why each change makes the plot more scientifically informative.

Possible improvements (choose any two — do not just pick the easiest ones):
- Add error bars if uncertainty data is available
- Add a horizontal line at bandgap = 0 to separate metals from semiconductors/insulators
- Label the 3 materials with the highest bandgap with their formula
- Add a marginal histogram on one axis to show the property distribution
- Color points by crystal system and add a legend
- Use a log scale on the x-axis and justify why

In [ ]:
# Cell E2 — Improved scatter plot
# Your code here:


**What I changed and why:** *(write here)*



### E3 — Retrieve and describe a material of your choice

**Task:** Choose any material relevant to your own research area and retrieve it from the Materials Project. It can be anything — an oxide, an alloy, a semiconductor, a 2D material — as long as it is in the MP database.

Then complete the analysis below.

In [ ]:
# Cell E3a — Retrieve your chosen material
# Replace 'mp-XXXX' with the actual Materials Project ID of your material.
# Find the ID by searching at https://materialsproject.org

MY_MATERIAL_ID = 'mp-XXXX'  # ← Replace this

with MPRester(API_KEY) as mpr:
    my_struct = mpr.get_structure_by_material_id(MY_MATERIAL_ID)
    my_results = mpr.summary.search(
        material_ids=[MY_MATERIAL_ID],
        fields=["formula_pretty", "band_gap", "formation_energy_per_atom",
                "energy_above_hull", "is_stable", "density",
                "volume", "nsites", "symmetry"]
    )

my_entry = my_results[0]

print("=" * 50)
print(f"Material ID:          {MY_MATERIAL_ID}")
print(f"Formula:              {my_entry.formula_pretty}")
print(f"Space group:          {my_struct.get_space_group_info()}")
print(f"Number of sites:      {my_entry.nsites}")
print(f"Volume:               {my_entry.volume:.3f} Å³")
print(f"Density:              {my_entry.density:.3f} g/cm³")
print(f"Band gap:             {my_entry.band_gap:.3f} eV")
print(f"Formation energy:     {my_entry.formation_energy_per_atom:.4f} eV/atom")
print(f"Energy above hull:    {my_entry.energy_above_hull:.4f} eV/atom")
print(f"Thermodynamic stable: {my_entry.is_stable}")
print("=" * 50)

In [ ]:
# Cell E3b — Nearest neighbor analysis for your material
# For each unique species in your structure, find its nearest neighbor
# and report the bond distance.

# Your code here:


### E3c — Written reflection

Answer the following three questions about your chosen material. Be specific — connect each answer to the actual values you retrieved above.

**1. Why did you choose this material?** What is its significance in your research area?

*(your answer here)*

---

**2. One property that surprises you or that you want to understand better.** Based on the values you retrieved, identify one property (bandgap, formation energy, stability, density, etc.) that is different from what you expected, or that you find interesting. Propose a physical explanation.

*(your answer here)*

---

**3. Structure-property connection.** Based on the crystal structure (space group, lattice parameters, coordination environment from E3b), identify one structural feature you think is responsible for a key property of this material. This does not need to be proven — it is a scientific hypothesis.

*(your answer here)*

---
## Submission Checklist

Before uploading to Canvas, confirm:

- [ ] Run `Kernel → Restart & Run All` — all cells execute without errors
- [ ] Cell A1: Python version ≥ 3.10 confirmed
- [ ] Cell A2: All imports successful
- [ ] Cell A3: MP API key loaded
- [ ] Cell A4: MP API connection test passed
- [ ] Part B: All three TASK cells have your code with output
- [ ] Part C: All cells run; C4 task has colorbar on scatter plot
- [ ] Part D: D4 uses a material ID different from mp-5020
- [ ] Part E: E1 written answer ≥ 2 sentences
- [ ] Part E: E2 improved plot with markdown explanation
- [ ] Part E: E3 uses your own chosen material (not mp-XXXX placeholder)
- [ ] Part E: E3c has written answers for all three questions
- [ ] AI disclosure note updated or deleted at the top of the notebook
- [ ] File renamed: `[LastName]_week1.ipynb`

**Due: Sunday 11:59 PM via Canvas**